In [ ]:
import sys
import os
import re
import joblib

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from slovene_pipeline.word2vec_api import Word2VecAPI, maybe_load_emoji2vec
from slovene_pipeline.features import build_tfidf, build_w2v_mean, combine_features
from irony_translation.LLMSarcasmTranslator import LLMSarcasmTranslator
from irony_translation.SarcasmTranslator import T5SarcasmTranslator

from long_text_parser.long_text_classification import classify_text, load_classifier

In [ ]:
MODEL_PATH = os.path.join(project_root, 'slovene_pipeline', 'subtaskA_notebooks', 'finalized_model.sav')

if os.path.exists(MODEL_PATH):
    irony_classifier = joblib.load(MODEL_PATH)
    if isinstance(irony_classifier, dict) and "vectorizer" in irony_classifier:
        vectorizer = irony_classifier["vectorizer"]
        clf = irony_classifier["classifier"]
    else:
        clf = irony_classifier
        vectorizer = None 
else:
    print(f"Model not found at {MODEL_PATH}")
    clf = None
    vectorizer = None

# Option 1: LLM Translator (Groq/OpenAI base)
API_KEY = "your-api-key-here" 
llm_translator = LLMSarcasmTranslator(api_key=API_KEY)

# Option 2: Local HuggingFace Translator (e.g., mT5 trained on dataset)
local_translator = T5SarcasmTranslator(model_name="csebuetnlp/mT5_multilingual_XLSum")
# Uncomment next line to load local weights if trained locally
# local_translator.load_model(os.path.join(project_root, "irony_translation", "from_small_dataset", "mT5_multilingual_XLSum"))

# Choose which one to pass to the pipeline
translator = llm_translator 
# translator = local_translator

In [ ]:
def split_sentences(text: str):
    parts = re.split(r'(?<=[.!?]) +', text.strip())
    sentences = [p.strip() for p in parts if p.strip()]
    return sentences

In [ ]:
def is_long_text_ironic(text: str, clf_model=None, threshold: float = 0.5) -> bool:
    if clf_model is None:
        return False
        
    try:
        prob = classify_text(text, clf_model)
        return prob >= threshold
    except Exception as e:
        print(f"Error during long text prediction: {e}")
        return False

def is_sentence_ironic(sentence: str, clf_model=None, vectorizer_model=None, w2v_model=None, emoji_model=None) -> bool:
    if clf_model is None:
        return False
        
    corpus = [sentence]
    try:
        tfidf_x, _ = build_tfidf(corpus, fitted_vectorizer=vectorizer_model)
        w2v_x = build_w2v_mean(corpus, w2v_model, emoji_model)
        x = combine_features(tfidf_x, w2v_x)
        
        preds = clf_model.predict(x)
        return bool(preds[0])
    except Exception as e:
        print(f"Error during sentence prediction: {e}")
        return False

In [ ]:
def translate_ironic_sentence(sentence: str, translator_model) -> str:
    try:
        if hasattr(translator_model, 'zero_shot'):
            # Used by LLMSarcasmTranslator
            non_ironic = translator_model.zero_shot(sentence)
        elif hasattr(translator_model, 'generate'):
            # Used by T5SarcasmTranslator
            non_ironic = translator_model.generate(sentence)
        else:
            return sentence
            
        return non_ironic.strip()
    except Exception as e:
        print(f"Translation failed: {e}")
        return sentence

In [ ]:
def process_long_text_pipeline(input_text: str, clf_model=None, vectorizer_model=None, w2v_model=None, emoji_model=None, translator_model=None):
    is_text_block_ironic = is_long_text_ironic(input_text, clf_model=clf_model)
    
    transformed_details = []
    final_sentences = []
    
    if not is_text_block_ironic:
        return {
            "original_text": input_text,
            "final_text": input_text,
            "is_block_ironic": False,
            "total_sentences": len(split_sentences(input_text)),
            "ironic_count": 0,
            "transformations": []
        }
        
    sentences = split_sentences(input_text)
    
    for i, sentence in enumerate(sentences):
        
        is_ironic = is_sentence_ironic(
            sentence, 
            clf_model=clf_model, 
            vectorizer_model=vectorizer_model, 
            w2v_model=w2v_model, 
            emoji_model=emoji_model
        )
        
        if is_ironic and translator_model is not None:
            translated_sentence = translate_ironic_sentence(sentence, translator_model)
            final_sentences.append(translated_sentence)
            
            transformed_details.append({
                "index": i,
                "original": sentence,
                "transformed": translated_sentence
            })
        else:
            final_sentences.append(sentence)
            
    final_string = " ".join(final_sentences)
    
    result = {
        "original_text": input_text,
        "final_text": final_string,
        "is_block_ironic": True,
        "total_sentences": len(sentences),
        "ironic_count": len(transformed_details),
        "transformations": transformed_details
    }
    
    return result

In [ ]:
sample_texts = [
    "To je bil res odličen dan. Komaj čakam, da ponovimo!",
    "Oh, super, spet dežuje ravno ko grem na morje. Zaprli so tudi mejo. Vrjem, da je to bil res 'odličen' dan.",
]

import json

for text in sample_texts:
    CLF = globals().get('clf', None)
    VEC = globals().get('vectorizer', None)
    W2V = globals().get('w2v', None)
    EMOJI = globals().get('emoji_model', None)
    TRANS = globals().get('translator', None)
    
    if CLF is None:
        def mock_long_text_ironic(t, *args, **kwargs):
            return "dežuje" in t.lower()
        
        def mock_sentence_ironic(s, *args, **kwargs):
            return "dežuje" in s.lower() or "odličen" in s.lower()
        
        original_is_long = is_long_text_ironic
        original_is_sent = is_sentence_ironic
        is_long_text_ironic = mock_long_text_ironic
        is_sentence_ironic = mock_sentence_ironic
        
    output = process_long_text_pipeline(text, clf_model=CLF, vectorizer_model=VEC, w2v_model=W2V, emoji_model=EMOJI, translator_model=TRANS)
    
    print("-" * 50)
    print(json.dumps(output, indent=2, ensure_ascii=False))
    
    if CLF is None:
        is_long_text_ironic = original_is_long
        is_sentence_ironic = original_is_sent

In [ ]:
import pandas as pd

def run_experiment_on_file_long(input_csv_path: str, output_csv_path: str):
    df = pd.read_csv(input_csv_path)
    
    CLF = globals().get('clf', None)
    VEC = globals().get('vectorizer', None)
    W2V = globals().get('w2v', None)
    EMOJI = globals().get('emoji_model', None)
    TRANS = globals().get('translator', None)
    
    results = []
    
    for index, row in df.iterrows():
        text = str(row['text'])
        true_label = row.get('true_label', '')
        
        output = process_long_text_pipeline(
            text, 
            clf_model=CLF, 
            vectorizer_model=VEC, 
            w2v_model=W2V, 
            emoji_model=EMOJI, 
            translator_model=TRANS
        )
        
        results.append({
            'input_text': text,
            'true_label': true_label,
            'predicted_ironic_block': output.get('is_block_ironic', False),
            'sentences_translated': output.get('ironic_count', 0),
            'output_text': output.get('final_text', text),
            'TP_FP_TN_FN': ''
        })
        
    out_df = pd.DataFrame(results)
    out_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
    print(f"Eksperiment zaključen. Podatki shranjeni v: {output_csv_path}")

# usage
# run_experiment_on_file_long("ročni_tviti.csv", "rezultati_eksperimenta_long.csv")